# Colab Environment Setup & Verification (P0-T2)
**Đề tài**: Nghiên cứu & Xây dựng mô hình All-in-One Image Restoration (AiOIR)
**Mục tiêu của notebook**:
1. Kết nối Google Drive để quản lý checkpoint và log lưu trữ lâu dài.
2. Kiểm tra phần cứng GPU T4 (~15GB VRAM) và cấu hình PyTorch/CUDA.
3. Thiết lập mã nguồn dự án trên đĩa cục bộ `/content/repo`.
4. Cài đặt các thư viện phụ thuộc từ `requirements.txt`.
5. Chạy thử nghiệm 1 batch giả lập (Dummy Batch Forward Pass) với Mixed Precision (AMP) để bảo đảm môi trường sẵn sàng 100%.

## 1. Kết Nối Google Drive (Mount Drive)

In [ ]:
from google.colab import drive
import os

# Mount Google Drive vào /content/drive
drive.mount('/content/drive')

# Tạo thư mục làm việc lưu trữ trên Drive nếu chưa có
drive_workspace = '/content/drive/MyDrive/nckh_coChau_workspace'
os.makedirs(drive_workspace, exist_ok=True)
print(f"Thư mục Drive đã sẵn sàng: {drive_workspace}")

## 2. Kiểm Tra GPU (NVIDIA-SMI) & Cấu Hình PyTorch / CUDA

In [ ]:
# Kiểm tra GPU được cấp phát bởi Google Colab
!nvidia-smi

import torch

print("="*50)
print(f"PyTorch Version : {torch.__version__}")
print(f"CUDA Available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    device_id = torch.cuda.current_device()
    gpu_name = torch.cuda.get_device_name(device_id)
    total_mem = torch.cuda.get_device_properties(device_id).total_memory / (1024**3)
    print(f"GPU Name        : {gpu_name}")
    print(f"VRAM Dung Lượng : {total_mem:.2f} GB")
    assert "T4" in gpu_name or total_mem >= 11.0, f"Cảnh báo: GPU {gpu_name} có thể không đủ VRAM tiêu chuẩn T4!"
    print(">> GPU & CUDA ĐÃ ĐẠT TIÊU CHUẨN!")
else:
    raise RuntimeError("LỖI: Chưa kích hoạt GPU! Hãy vào Runtime -> Change runtime type -> Chọn T4 GPU.")
print("="*50)

## 3. Thiết Lập Mã Nguồn Repo Tại Cục Bộ `/content/repo`

In [ ]:
import os
import shutil

# Thiết lập thư mục repo trên đĩa SSD cục bộ để tăng tốc tối đa I/O
local_repo_dir = '/content/repo'

# Nếu bạn đã đồng bộ repo lên Drive:
drive_repo_dir = '/content/drive/MyDrive/nckh_coChau/repo'

if os.path.exists(drive_repo_dir):
    print(f"Tìm thấy repo trên Drive tại {drive_repo_dir}. Đang copy sang đĩa cục bộ...")
    if os.path.exists(local_repo_dir):
        shutil.rmtree(local_repo_dir)
    shutil.copytree(drive_repo_dir, local_repo_dir)
    print("Đã copy repo sang /content/repo!")
else:
    # Trường hợp clone trực tiếp từ GitHub (thay link repo GitHub của bạn khi sẵn sàng):
    print("Chưa có bản copy trên Drive, kiểm tra thư mục hiện tại hoặc clone từ Git...")
    os.makedirs(local_repo_dir, exist_ok=True)

%cd /content/repo
!pwd
!ls -la

## 4. Cài Đặt Các Gói Thư Viện Phụ Thuộc Từ `requirements.txt`

In [ ]:
# Cài đặt các thư viện phụ thuộc (einops, timm, lpips, ptflops, scikit-image...)
# Tận dụng PyTorch và Torchvision đã cài sẵn của Colab
if os.path.exists('requirements.txt'):
    !pip install -r requirements.txt
else:
    print("Cài đặt trực tiếp danh sách thư viện chuẩn...")
    !pip install einops>=0.6.0 timm>=0.9.0 lpips>=0.1.4 ptflops>=0.7.0 scikit-image pyyaml tqdm

print(">> Đã cài đặt hoàn tất các gói thư viện!")

## 5. Chạy Kiểm Thử 1 Dummy Batch Với Mixed Precision (AMP) Trên GPU

In [ ]:
import torch
import torch.nn as nn
from torch.cuda.amp import autocast, GradScaler

print(">> Bắt đầu kiểm thử Dummy Batch trên GPU...")

# Định nghĩa một khối phục hồi ảnh tối giản (SimpleGate + Conv + Residual)
class DummyRestorationBlock(nn.Module):
    def __init__(self, channels=32):
        super().__init__()
        self.conv1 = nn.Conv2d(3, channels, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(channels, channels * 2, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(channels, channels, kernel_size=3, padding=1)
        self.out_conv = nn.Conv2d(channels, 3, kernel_size=3, padding=1)
        
    def forward(self, x):
        feat = self.conv1(x)
        # Cơ chế SimpleGate (NAFNet style): chia đôi kênh và nhân điểm chập
        gate_x = self.conv2(feat)
        x1, x2 = gate_x.chunk(2, dim=1)
        gated = x1 * x2
        feat = feat + self.conv3(gated)
        return x + self.out_conv(feat)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = DummyRestorationBlock(channels=48).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
criterion = nn.L1Loss()
scaler = GradScaler() # Hỗ trợ AMP cho Colab T4

# Tạo 1 dummy batch: batch_size=2, channels=3, height=128, width=128 (chuẩn patch size Colab Free)
dummy_input = torch.randn(2, 3, 128, 128, device=device)
dummy_target = torch.randn(2, 3, 128, 128, device=device)

# Chạy thử nghiệm 1 forward & backward pass có AMP
optimizer.zero_grad()
with autocast():
    output = model(dummy_input)
    loss = criterion(output, dummy_target)

scaler.scale(loss).backward()
scaler.step(optimizer)
scaler.update()

vram_used = torch.cuda.max_memory_allocated() / (1024**2)
print(f"[THÀNH CÔNG] Forward & Backward pass hoàn tất không lỗi!")
print(f"- Input Shape    : {dummy_input.shape}")
print(f"- Output Shape   : {output.shape}")
print(f"- Loss Giá Trị   : {loss.item():.4f}")
print(f"- Peak VRAM Used : {vram_used:.2f} MB")
print("="*50)
print(">> MÔI TRƯỜNG COLAB ĐÃ SẴN SÀNG CHO TASK P1 VÀ CÁC THÍ NGHIỆM TIẾP THEO!")